# Sentinel airline volatility model (Colab)

This notebook runs **the same model code as the website's daily update**. The maths lives in
`pipeline/sentinel_model.py` in the [sentinel-live](https://github.com/CrunchyBoba/sentinel-live) repository;
the notebook clones it and imports it, so the two can never drift apart.

**You don't need to run this to keep the site up to date.** A GitHub Action re-runs the models after every
trading day and publishes the results. Use this notebook to explore, plot and experiment.

Steps:
1. Download prices for the 8 airlines
2. HMM regimes (Calm / Stressed / Crisis)
3. Forecast the next ~2 months (42 trading days) with EGARCH and TimesFM-3 (alone and with the sector)
4. PCA: sector vs stock-specific risk
5. The track record: how past forecasts compared with what happened
6. (Optional) run the full daily pipeline and download the site's data file

In [ ]:
%pip install -q yfinance pandas numpy arch hmmlearn scikit-learn matplotlib
# TimesFM-3, pinned to the same commit as the daily pipeline
%pip install -q "timesfm[torch] @ git+https://github.com/google-research/timesfm.git@e51928e"

!rm -rf sentinel-live && git clone -q --depth 1 https://github.com/CrunchyBoba/sentinel-live.git
import sys, json
sys.path.insert(0, 'sentinel-live/pipeline')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import sentinel_model as M

print('Horizon:', M.HORIZON, 'trading days · tickers:', M.TICKERS)

## 1. Data
Returns are daily log returns ×100 (percent). ULCC started trading in 2021; each airline uses the data it has.

In [ ]:
prices, volume = M.download_prices()
returns, sector, vol_df, sector_vol = M.make_returns(prices)
print(returns.index[0].date(), 'to', returns.index[-1].date(), '|', len(returns), 'days')

## 2. HMM regimes
The HMM sorts days into 3 hidden states by volatility: **Calm**, **Stressed**, **Crisis**.

In [ ]:
hmm, order = M.fit_hmm(sector)
probs, labels, trans = M.get_regimes(hmm, order, sector)
print('Today:', dict(zip(M.REGIMES, probs[-1].round(3))))
print(pd.DataFrame(trans, index=M.REGIMES, columns=M.REGIMES).round(3))
pd.DataFrame(probs, index=sector.index, columns=M.REGIMES).tail(500).plot.area(figsize=(14, 3.5), color=['green', 'orange', 'red'], alpha=.7)
plt.title('HMM regime probabilities (last 500 days)'); plt.show()

## 3. Forecasts: next ~2 months
- **EGARCH(1,1,1)**: volatility clustering, with bad-news days raising risk more than good-news days. If `arch`'s
  multi-day simulation looks broken for a stock, a direct simulation of the same equation is used (printed below).
- **TimesFM-3**: Google's pre-trained forecaster, run zero-shot on each airline's 20-day volatility series, alone
  and jointly with the sector. Weights are free for research and non-commercial use only.

In [ ]:
tfm = M.TimesFM()
paths, backups = M.forecast_all(returns, vol_df, sector_vol, tfm)
if backups: print('EGARCH backup simulation used for:', backups)
future = pd.bdate_range(returns.index[-1] + pd.Timedelta(days=1), periods=M.HORIZON)

fig, axes = plt.subplots(4, 2, figsize=(14, 14))
for ax, t in zip(axes.flatten(), M.TICKERS):
    recent = vol_df[t].dropna().tail(120)
    ax.plot(recent.index, recent.values, color='black', label='Recent (20-day)')
    ax.plot(future, paths['egarch'][t], '--', color='#b8893a', label='EGARCH')
    ax.plot(future, paths['timesfm_alone'][t], ':', color='#1d6f73', label='TimesFM')
    ax.plot(future, paths['timesfm_together'][t], ':', color='#5aa596', label='TimesFM + sector')
    ax.set_title(t); ax.set_ylabel('Annualized vol (%)')
axes[0, 0].legend(); plt.tight_layout(); plt.show()

## 4. PCA: sector vs stock-specific risk

In [ ]:
aligned = returns.dropna()
share = M.sector_share(aligned.tail(M.PCA_WINDOW))
M.pca_history(aligned).plot(figsize=(14, 4), alpha=.7); plt.title('Share of risk explained by the sector (rolling)'); plt.show()
rms = lambda p: float(np.sqrt(np.mean(np.asarray(p) ** 2)))
final = pd.DataFrame({t: {'EGARCH %': rms(paths['egarch'][t]), 'TimesFM %': rms(paths['timesfm_alone'][t]),
                          'TimesFM + sector %': rms(paths['timesfm_together'][t]), 'Sector share': share[t]} for t in M.TICKERS}).T
final.round(2).sort_values('EGARCH %', ascending=False)

## 5. Track record
The daily job logs every forecast in `data/forecast_log.csv` before its outcome exists, and scores it later:
- **Next day**: did the next day's move land inside the forecast's 95% range? (about 95% should)
- **1 week / 2 months**: forecast volatility vs the volatility that actually happened.

Rows marked `reconstructed` were rebuilt walk-forward (only prices available on each day); `live` rows were logged by the daily run.

In [ ]:
log = pd.read_csv('sentinel-live/data/forecast_log.csv')
print(log['source'].value_counts())
site = json.load(open('sentinel-live/data/sentinel_data.json'))
rows = []
for m, w in site.get('track', {}).get('sector', {}).items():
    a = w.get('all', {})
    rows.append({'model': m, 'next-day inside %': 100 * a.get('day', {}).get('hit', np.nan),
                 '1w avg miss (pts)': a.get('w1', {}).get('mae'), '1w bias': a.get('w1', {}).get('bias'),
                 '2m avg miss (pts)': a.get('m2', {}).get('mae'), '2m bias': a.get('m2', {}).get('bias')})
pd.DataFrame(rows).set_index('model').round(2)

## 6. (Optional) run the full daily pipeline here
This does exactly what the GitHub Action does and writes `data/sentinel_data.json` inside the clone.
You normally don't need it; the Action publishes automatically. `--force` re-runs even if today's close is already logged.

In [ ]:
!cd sentinel-live && python pipeline/update_model.py --force
from google.colab import files
files.download('sentinel-live/data/sentinel_data.json')